# Visual Analytics

## Assignment 3

**Instructor:** Dr. Marco D'Ambros  
**TAs:** Giuseppe Crupi, Mattia Giannaccari

**Contacts:** marco.dambros@usi.ch, giuseppe.crupi@usi.ch, mattia.giannaccari@usi.ch

**Due Date:** May 25, 2026 @ 23:55

---
The goal of this assignment is to use **Spark (PySpark)** and **Polars** in Jupyter notebooks.  
The files `trip_data.csv`, `trip_fare.csv`, and `nyc_boroughs.geojson` are available in the provided folder: [Assignment3-data](https://usi365-my.sharepoint.com/:f:/g/personal/armenc_usi_ch/Ejp7sb8QAMROoWe0XUDcAkMBoqUFk-w2Vgroup025NhAww?e=2I7SMC).

- Use **Spark** to solve **Exercises 1–4**
- Use **Polars** to solve **Exercises 5–8**

Please name your notebook file as `SurnameName_Assignment3.ipynb`

# ⚡️ Spark Exercises (50 pts)

In [2]:
from pyspark.sql import SparkSession
import polars as pl
from pathlib import Path
import os


# os.environ() sets the env variable for java jdk at the path of jdk you want to use for spark, only for the current session
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

print(os.environ.get("JAVA_HOME"))

CWD = Path.cwd()
DATA = "data"
TRIP_DATA = "trip_data.csv"
TRIP_FARE = "trip_fare.csv"
NYC_BOROUGHS = "nyc-boroughs.geojson"
DATA_ROOT = CWD / DATA
TRIP_DATA_ROOT = DATA_ROOT / TRIP_DATA
TRIP_FARE_ROOT = DATA_ROOT / TRIP_FARE
NYC_BOROUGHS_ROOT = DATA_ROOT / NYC_BOROUGHS

/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home


### Exercise 1 (8 pts)
Join the `trip_data` and `trip_fare` dataframes into one, considering only trips from January 1–7, 2013. Filter out trips where the total amount charged is 0 or less, or the trip distance is 0 or less. Report how many rows are removed by each filter, the total number of rows after filtering, and the average tip amount across the resulting dataset.

In [3]:
spark = SparkSession.builder \
    .appName("Assignment3") \
    .getOrCreate()
trip_data_df = spark.read.csv(str(TRIP_DATA_ROOT), header=True, inferSchema=True)
print("TRIP DATA DATA SCHEMA")
trip_data_df.printSchema()
print(f"TOTAL ROWS OF TRIP_DATA: {trip_data_df.count()}")
trip_fare_df = spark.read.csv(str(TRIP_FARE_ROOT), header=True, inferSchema=True)
print("TRIP FARE DATA SCHEMA")
trip_fare_df.printSchema()
print(f"TOTAL ROWS OF TRIP_FARE: {trip_fare_df.count()}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 18:08:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TRIP DATA DATA SCHEMA
root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- rate_code: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_time_in_secs: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)



TOTAL ROWS OF TRIP_DATA: 14776615


TRIP FARE DATA SCHEMA
root
 |-- medallion: string (nullable = true)
 |--  hack_license: string (nullable = true)
 |--  vendor_id: string (nullable = true)
 |--  pickup_datetime: timestamp (nullable = true)
 |--  payment_type: string (nullable = true)
 |--  fare_amount: double (nullable = true)
 |--  surcharge: double (nullable = true)
 |--  mta_tax: double (nullable = true)
 |--  tip_amount: double (nullable = true)
 |--  tolls_amount: double (nullable = true)
 |--  total_amount: double (nullable = true)



TOTAL ROWS OF TRIP_FARE: 14776615


In [4]:
from pyspark.sql.functions import col
# Now you can run your cleanup
trip_data_df = trip_data_df.select([col(c).alias(c.strip()) for c in trip_data_df.columns])
trip_fare_df = trip_fare_df.select([col(c).alias(c.strip()) for c in trip_fare_df.columns])

# Perform the join on the 4 common primary keys
join_keys = ["medallion", "hack_license", "vendor_id", "pickup_datetime"]
trip_data_fare_df = trip_data_df.join(trip_fare_df, on=join_keys, how="inner")
# Verify the count and schema
total_rows = trip_data_fare_df.count()
print(f"TOTAL ROWS AFTER JOIN: {total_rows}")
trip_data_fare_df.printSchema()

TOTAL ROWS AFTER JOIN: 14776615
root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- rate_code: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_time_in_secs: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [5]:
from pyspark.sql.functions import col, to_date, avg

date_filtered_df = trip_data_fare_df.filter(
    (to_date(col("pickup_datetime")).between("2013-01-01", "2013-01-07")))
total_date_filtered_rows = date_filtered_df.count()
print(f"#ROWS REMOVED FOR 2013-01-01 to 2013-01-07 FILTER: {total_rows - total_date_filtered_rows}")

total_amount_fare_filtered_df = date_filtered_df.filter(
    ~((col("total_amount")) <= 0))
total_amount_fare_filtered_rows = total_amount_fare_filtered_df.count()
print(f"#ROWS REMOVED FOR TOTAL AMOUNT <= 0 FILTER: {total_date_filtered_rows - total_amount_fare_filtered_rows}")

trip_distance_filtered_df = total_amount_fare_filtered_df.filter(
    ~(col("trip_distance") <= 0))
total_trip_distance_filtered_rows = trip_distance_filtered_df.count()
print(f"#ROWS REMOVED FOR TRIP DISTANCE <= 0 FILTER: {total_amount_fare_filtered_rows - total_trip_distance_filtered_rows}")

print(f"#ROWS AFTER FILTERING: {total_trip_distance_filtered_rows}")

avg_tip_amount = trip_distance_filtered_df.select(avg("tip_amount")).collect()[0][0]
print(f"AVG TIP AMOUNT AFTER FILTERING: {avg_tip_amount}")

#ROWS REMOVED FOR 2013-01-01 to 2013-01-07 FILTER: 11766480


#ROWS REMOVED FOR TOTAL AMOUNT <= 0 FILTER: 0


#ROWS REMOVED FOR TRIP DISTANCE <= 0 FILTER: 19645
#ROWS AFTER FILTERING: 2990490


AVG TIP AMOUNT AFTER FILTERING: 1.1296970095201544


### Summary of Data Processing (PySpark)

| Metric | Value |
| :--- | :--- |
| **Rows removed (Date Filter: Jan 1–7, 2013)** | 11,766,480 |
| **Rows removed (Total Amount <= 0)** | 0 |
| **Rows removed (Trip Distance <= 0)** | 19,645 |
| **Total rows remaining after filtering** | **2,990,490** |
| **Average Tip Amount** | **$1.13** |

---
*Note: The total amount filter resulted in 0 removals, indicating that after the date filtering, all remaining records had a positive total amount charged.*


### Exercise 2 (12 pts)
For each hour of the day (0–23), compute the average trip duration in minutes and the average trip distance. Provide a graphical representation that allows comparing both metrics across hours side by side. You may want to have a look at: https://docs.bokeh.org/en/latest/docs/user_guide/basic/bars.html#grouping

In [6]:
from pyspark.sql.functions import hour, col, avg

useful_columns = ["trip_time_in_secs", "trip_distance", "pickup_datetime"]

# Using the filtered dataframe from the previous step to ensure data quality
ex2_cols_df = trip_distance_filtered_df.select(useful_columns)

# Convert seconds to minutes
ex2_cols_df = ex2_cols_df.withColumn(
    "trip_time_in_mins", 
    col("trip_time_in_secs") / 60
)

# Group by hour and calculate averages
avg_distance_trip_time_df = ex2_cols_df.groupBy(
    hour("pickup_datetime").alias("day_hour")
).agg(
    avg("trip_time_in_mins").alias("avg_trip_time"),
    avg("trip_distance").alias("avg_trip_distance")
).orderBy("day_hour")

# Show results for all 24 hours
avg_distance_trip_time_df.show(24)


+--------+------------------+------------------+
|day_hour|     avg_trip_time| avg_trip_distance|
+--------+------------------+------------------+
|       0|11.223678905155811|3.1824580652573533|
|       1| 11.41462029566623|3.1878349602436935|
|       2|11.304379693818598| 3.266443770460238|
|       3|11.261855373762797|3.5235907953821557|
|       4| 11.76209852736189| 4.130701579528959|
|       5|12.035606561144464| 4.981900548677144|
|       6|10.405564935503483|3.7657435672317168|
|       7|10.550317880700407|3.1463314146581105|
|       8|11.212508254337282|2.7732620049444177|
|       9|10.773916100997642|2.6097442147485332|
|      10|10.651816069479327|  2.71362143515722|
|      11|10.621733516892393|2.6400796249441862|
|      12|10.682610332471544| 2.611790855898762|
|      13|11.189539924439162| 2.755013655976754|
|      14|11.869509862600177|2.9462623607786007|
|      15| 11.82622066408344| 2.944925007248856|
|      16|11.368703959982124| 2.880618503942683|
|      17|11.6091186

In [ ]:
# Collect the data to the driver (since it's only 24 rows)
results = avg_distance_trip_time_df.collect()

# Use list comprehension to extract specific columns
list_avg_time = [row["avg_trip_time"] for row in results]
list_avg_distance = [row["avg_trip_distance"] for row in results]
list_hours = [row["day_hour"] for row in results]



In [ ]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import column
from bokeh.models import ColumnDataSource

x_axis_label = "Hour of Day (in range 2013-01-01 to 2013-01-07)"

# Enable bokeh to display inside the Jupyter Notebook
output_notebook()

# Prepare the data source
source = ColumnDataSource(data=dict(
    hours=[str(h) for h in list_hours], # Convert to string for discrete x-axis
    avg_time=list_avg_time,
    avg_dist=list_avg_distance
))

# Create the Average Trip Time plot
p1 = figure(x_range=[str(h) for h in list_hours], height=350, title="Average Trip Time per Hour",
           toolbar_location=None, tools="", x_axis_label=x_axis_label, y_axis_label="Minutes")

p1.vbar(x='hours', top='avg_time', width=0.9, source=source, color="#3182bd")
p1.xgrid.grid_line_color = None
p1.y_range.start = 0

# Create the Average Trip Distance plot
p2 = figure(x_range=[str(h) for h in list_hours], height=350, title="Average Trip Distance per Hour",
           toolbar_location=None, tools="", x_axis_label=x_axis_label, y_axis_label="Miles")

p2.vbar(x='hours', top='avg_dist', width=0.9, source=source, color="#e6550d")
p2.xgrid.grid_line_color = None
p2.y_range.start = 0

# Display both plots in a column layout
show(column(p1, p2))


Loading BokehJS ...

### Exercise 3 (14 pts)
Consider only the boroughs Queens, Staten Island, and EWR. Create a dataframe that shows, for each payment type, the total fare amount collected for trips *originating from* each of those three boroughs, broken down by *destination borough* (including all boroughs as destinations).

> For example, for Queens you should consider:
> - Queens → Queens (cash), Queens → Queens (card), ...
> - Queens → Manhattan (cash), Queens → Manhattan (card), ...
> - and so on for all destination boroughs.


In [9]:
import json
from shapely.geometry import shape, MultiPolygon
from shapely.ops import unary_union
from shapely.prepared import prep

with open("data/nyc-boroughs.geojson") as f:
    geojson_data = json.load(f)

# Group features by borough name
borough_polygons = {}
for feature in geojson_data['features']:
    name = feature['properties']['borough']
    geometry = shape(feature['geometry'])
    
    if name not in borough_polygons:
        borough_polygons[name] = []
    borough_polygons[name].append(geometry)

# Combine list of polygons into a single MultiPolygon for each borough
# unary_union is very efficient at merging multiple shapes
boroughs_map = {}
for name, polygons in borough_polygons.items():
    boroughs_map[name] = unary_union(polygons)

print(f"Mapped {len(boroughs_map)} boroughs: {list(boroughs_map.keys())}")



Mapped 5 boroughs: ['Staten Island', 'Queens', 'Brooklyn', 'Manhattan', 'Bronx']


In [11]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
from shapely.geometry import Point

# We use the standard 'boroughs_map' instead of the prepared version
def find_borough(lon, lat):
    if lon is None or lat is None:
        return "Unknown"
    
    p = Point(lon, lat)
    # Check each borough's standard MultiPolygon
    for name, geom in boroughs_map.items():
        if geom.contains(p):
            return name
    return "Unknown"

# Register UDF
find_borough_udf = udf(find_borough, StringType())


In [12]:
from pyspark.sql.functions import col, sum

# 1. Create a projection of the required columns
selected_cols = [
    "payment_type", 
    "pickup_longitude", 
    "pickup_latitude", 
    "dropoff_longitude", 
    "dropoff_latitude", 
    "fare_amount"
]

# We use the filtered dataframe to ensure we only process valid trips
proj_df = trip_distance_filtered_df.select(selected_cols)

# Create new columns for pickup and dropoff boroughs using the UDF
trips_with_boroughs_df = proj_df \
    .withColumn("pickup_borough", find_borough_udf(col("pickup_longitude"), col("pickup_latitude"))) \
    .withColumn("dropoff_borough", find_borough_udf(col("dropoff_longitude"), col("dropoff_latitude")))

# Group by payment_type, pickup_borough, and dropoff_borough, then sum the fare_amount
borough_fare_stats = trips_with_boroughs_df.groupBy(
    "payment_type", 
    "pickup_borough", 
    "dropoff_borough"
).agg(
    sum("fare_amount").alias("total_fare_amount")
).orderBy(
    "payment_type", "pickup_borough", col("total_fare_amount").desc() 
)

borough_list = ["Queens", "Staten Island", "EWR"]

filtered_stats = borough_fare_stats.filter(
    col("pickup_borough").isin(borough_list)
).orderBy("pickup_borough")

# Show the results
filtered_stats.show(50, truncate=False)



+------------+--------------+---------------+------------------+
|payment_type|pickup_borough|dropoff_borough|total_fare_amount |
+------------+--------------+---------------+------------------+
|UNK         |Queens        |Manhattan      |3236.5            |
|CRD         |Queens        |Bronx          |31561.5           |
|CSH         |Queens        |Bronx          |74741.5           |
|CSH         |Queens        |Queens         |507712.77         |
|DIS         |Queens        |Manhattan      |2246.3            |
|NOC         |Queens        |Queens         |4496.0            |
|CSH         |Queens        |Unknown        |55404.97          |
|CSH         |Queens        |Brooklyn       |405903.0          |
|UNK         |Queens        |Unknown        |666.5             |
|CSH         |Queens        |Staten Island  |7273.0            |
|NOC         |Queens        |Unknown        |352.0             |
|CRD         |Queens        |Brooklyn       |465718.5          |
|DIS         |Queens     

### Exercise 4 (16 pts)
Create a dataframe where each row represents a driver, and there is one column per hour of the day (0–23). For each driver-hour, the dataframe provides the maximum number of consecutive trips where the tip amount was strictly greater than $0.

> For example, if for driver B we have trips starting in hour 14 (sorted by pickup time):
>
> - Trip 1: tip = $2.00
> - Trip 2: tip = $0.00
> - Trip 3: tip = $1.50
> - Trip 4: tip = $3.00
>
> The longest streak of tipped trips in hour 14 is 2 (Trips 3 and 4).

Additionally, print the pair (driver, hour) with the maximum streak.

In [14]:

from pyspark.sql import Window
from pyspark.sql.functions import hour, col, lag, sum as _sum, count, max as _max, when

df_streaks = trip_distance_filtered_df.select("hack_license", "pickup_datetime", "tip_amount") \
    .withColumn("hour", hour("pickup_datetime")) \
    .withColumn("has_tip", when(col("tip_amount") > 0, 1).otherwise(0))

w = Window.partitionBy("hack_license", "hour").orderBy("pickup_datetime")

df_streaks = df_streaks.withColumn("prev_has_tip", lag("has_tip").over(w)) \
    .withColumn("streak_start", when((col("has_tip") == 1) & ((col("prev_has_tip") == 0) | col("prev_has_tip").isNull()), 1).otherwise(0))
# Create a unique ID for each streak using cumulative sum
df_streaks = df_streaks.withColumn("streak_id", _sum("streak_start").over(w))

final_streaks = df_streaks.filter(col("has_tip") == 1) \
    .groupBy("hack_license", "hour", "streak_id") \
    .agg(count("*").alias("streak_len")) \
    .groupBy("hack_license", "hour") \
    .agg(_max("streak_len").alias("max_streak"))
# PIVOT: This creates one column per hour (0-23)
result_pivot_df = final_streaks.groupBy("hack_license") \
    .pivot("hour", range(24)) \
    .agg(_max("max_streak")) \
    .fillna(0) # Replace nulls with 0 for hours with no streaks
result_pivot_df.show(1000)

top_driver = final_streaks.orderBy(col("max_streak").desc()).limit(1)

top_driver.show()


+--------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
|        hack_license|  0|  1|  2|  3|  4|  5|  6|  7|  8|  9| 10| 11| 12| 13| 14| 15| 16| 17| 18| 19| 20| 21| 22| 23|
+--------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
|01606C9E10D8D0B19...|  1|  1|  0|  0|  0|  1|  3|  1|  0|  1|  1|  1|  0|  1|  1|  1|  3|  3|  2|  5|  3|  1|  3|  4|
|02548BECEDACA82F0...|  2|  1|  2|  4|  1|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  2|  4|  5|  2|  2|  1|  3|
|02856AFC22881ABCA...|  2|  4|  1|  2|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  2|  2|  8|  5|  2|  4|  2|
|02E3C1D2FE5D53C22...|  0|  0|  0|  0|  4|  2|  0|  2|  4|  2|  3|  1|  1|  2|  9|  4|  0|  0|  0|  0|  0|  0|  0|  0|
|03A2D28F831C5C3E5...|  0|  0|  0|  0|  0|  0|  0|  0|  3|  3|  2|  2|  3|  2|  6|  3|  3|  2|  0|  0|  0|  0|  0|  0|
|0DC7C87D535512AF8...|  0|  0|  0|  0|  0|  0|  

+--------------------+----+----------+
|        hack_license|hour|max_streak|
+--------------------+----+----------+
|F9E822E2938FCE35E...|  20|        18|
+--------------------+----+----------+



# 🐻‍❄️ Polars Exercises (50 pts)

In this section, you will use **Polars** to perform data cleaning, transformation, and analysis on the NYC taxi dataset.

You will work with the merged dataset obtained from:
- `trip_data.csv`
- `trip_fare.csv`

### Exercise 5 (10 pts)

Perform a sequence of data cleaning steps on the dataset:

1. Remove trips where:
   - `trip_distance <= 10` but `fare_amount > 100`.
   - `trip_distance > 100` or `trip_distance <= 0>` miles.

2. Remove trips with:
   - missing timestamps (`pickup_datetime`, `dropoff_datetime`).
   - `dropoff_datetime <= pickup_datetime`.

After each step:
- Report how many rows were removed.

Finally:
- Report the number of remaining rows.
- Check whether duplicate records exist (based on `medallion`, `hack_license`, `pickup_datetime`).

In [15]:
df_trips = pl.read_csv(str(TRIP_DATA_ROOT), infer_schema_length=10000).rename(lambda s: s.strip())
df_fares = pl.read_csv(str(TRIP_FARE_ROOT), infer_schema_length=10000).rename(lambda s: s.strip())

df_combined = df_trips.join(
    df_fares, 
    on=["medallion", "hack_license", "vendor_id", "pickup_datetime"], 
    how="inner"
)

df_combined = df_combined.with_columns(
    pl.col("pickup_datetime").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("dropoff_datetime").str.to_datetime("%Y-%m-%d %H:%M:%S")
)

df_combined.glimpse()

Rows: 14776615
Columns: 21
$ medallion                   <str> '89D227B655E5C82AECF13C3F540D4CF4', '0BD7C8F5BA12B88E0B67BED28BEA73D8', '0BD7C8F5BA12B88E0B67BED28BEA73D8', 'DFD2202EE08F7A8DC9A57B02ACB81FE2', 'DFD2202EE08F7A8DC9A57B02ACB81FE2', '20D9ECB2CA0767CF7A01564DF2844A3E', '496644932DF3932605C22C7926FF0FE0', '0B57B9633A2FECD3D3B1944AFC7471CF', '2C0E91FF20A856C891483ED63589F982', '2D4B95E2FA7B2E85118EC5CA4570FA58'
$ hack_license                <str> 'BA96DE419E711691B9445D6A6307C170', '9FD8F69F0804BDB5549F40E9DA1BE472', '9FD8F69F0804BDB5549F40E9DA1BE472', '51EE87E3205C985EF8431D850C786310', '51EE87E3205C985EF8431D850C786310', '598CCE5B9C1918568DEE71F43CF26CD2', '513189AD756FF14FE670D10B92FAF04C', 'CCD4367B417ED6634D986F573A552A62', '1DA2F6543A62B8ED934771661A9D2FA0', 'CD2F522EEE1FF5F5A8D8B679E23576B3'
$ vendor_id                   <str> 'CMT', 'CMT', 'CMT', 'CMT', 'CMT', 'CMT', 'CMT', 'CMT', 'CMT', 'CMT'
$ rate_code                   <i64> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1
$ store_and_f

In [16]:

count_start = df_combined.height

bad_trips_1 = (
    ((pl.col("trip_distance") <= 10) & (pl.col("fare_amount") > 100)) | 
    ((pl.col("trip_distance") > 100) | (pl.col("trip_distance") <= 0))
)

df_step1 = df_combined.filter(~bad_trips_1)
removed_step1 = count_start - df_step1.height

print(F"ROWS REMOVED AFTER STEP 1: {removed_step1}")

bad_trips_2 = (
    pl.col("pickup_datetime").is_null() | 
    pl.col("dropoff_datetime").is_null() | 
    (pl.col("dropoff_datetime") <= pl.col("pickup_datetime"))
)


df_filtered = df_step1.filter(~bad_trips_2)
removed_step2 = df_step1.height - df_filtered.height

print(f"ROWS REMOVED AFTER STEP 2: {removed_step2}")

print(f"TOTAL #ROWS REMOVED: {removed_step1 + removed_step2}")
print(f"TOTAL #ROWS REMAINING: {df_filtered.height}")

num_duplicated = df_filtered.height - df_filtered.unique(subset=["medallion", "hack_license", "pickup_datetime"]).height
print(f"THERE ARE {num_duplicated} DUPLICATED RECORDS")




ROWS REMOVED AFTER STEP 1: 83896
ROWS REMOVED AFTER STEP 2: 10030
TOTAL #ROWS REMOVED: 93926
TOTAL #ROWS REMAINING: 14682689
THERE ARE 0 DUPLICATED RECORDS


### Data Cleaning Results (Exercise 5)

| Metric | Value |
| :--- | :--- |
| **Total Rows Removed** | 93,926 |
| **Total Rows Remaining** | 14,682,689 |
| **Duplicate Records Found** | 0 |


### Exercise 6 (12 pts)

Analyze temporal patterns in taxi demand:

1. Group the data by `(weekday, hour)` and compute:
   - total number of trips
   - average fare per trip

2. Visualize the results using a **heatmap**

3. Return the top 5 `(weekday, hour)` by average fare.

In [18]:

weekday_map = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}

ex6_df = df_filtered.group_by(
pl.col("pickup_datetime").dt.weekday().alias("weekday"),
pl.col("pickup_datetime").dt.hour().alias("day_hour")) \
    .agg(
        pl.len().alias("total_trips"),
        pl.col("fare_amount").mean().alias("avg_fare_amount")
    ).sort("weekday", "day_hour")

plot_df = ex6_df.with_columns(
    pl.col("weekday").cast(pl.String).replace(weekday_map)
)

top_5_fares = plot_df.sort("avg_fare_amount", descending=True).head(5)
print(top_5_fares)


shape: (5, 4)
┌─────────┬──────────┬─────────────┬─────────────────┐
│ weekday ┆ day_hour ┆ total_trips ┆ avg_fare_amount │
│ ---     ┆ ---      ┆ ---         ┆ ---             │
│ str     ┆ i8       ┆ u32         ┆ f64             │
╞═════════╪══════════╪═════════════╪═════════════════╡
│ Mon     ┆ 5        ┆ 16792       ┆ 17.037845       │
│ Sat     ┆ 5        ┆ 17184       ┆ 16.582082       │
│ Fri     ┆ 5        ┆ 18182       ┆ 16.492713       │
│ Sun     ┆ 6        ┆ 17068       ┆ 16.205944       │
│ Mon     ┆ 4        ┆ 11775       ┆ 16.106072       │
└─────────┴──────────┴─────────────┴─────────────────┘


In [19]:
import hvplot.polars

# 2. Heatmap 1: Total Trips (REVERSED color and DISABLED scroll)
heatmap_trips = plot_df.hvplot.heatmap(
    x='weekday', 
    y='day_hour', 
    C='total_trips', 
    cmap='Magma',
    title='Total Number of Trips',
    xlabel='Day of Week',
    ylabel='Hour of Day',
    width=450,
    height=500
).opts(active_tools=[])

# 3. Heatmap 2: Average Fare Amount (DISABLED scroll)
heatmap_fares = plot_df.hvplot.heatmap(
    x='weekday', 
    y='day_hour', 
    C='avg_fare_amount', 
    cmap='Magma',
    title='Average Fare Amount',
    xlabel='Day of Week',
    ylabel='Hour of Day',
    width=450,
    height=500
).opts(active_tools=[])

# Display them side-by-side
(heatmap_trips + heatmap_fares).cols(2)


%opts magic unavailable (pyparsing cannot be imported)
%compositor magic unavailable (pyparsing cannot be imported)


:Layout
   .HeatMap.I  :HeatMap   [weekday,day_hour]   (total_trips)
   .HeatMap.II :HeatMap   [weekday,day_hour]   (avg_fare_amount)

### Exercise 7 (12 pts)

Define a *high-value trip* as one satisfying **at least two** of the following conditions:

- `fare_amount` is in the top 10%
- `tip_amount > 50%` of `fare_amount`
- `trip_distance < 2 miles` AND `fare_amount` above the median

Tasks:

1. Extract all high-value trips.
2. Select only the rides longer than 10 miles (in a straight line).
3. Report the total number of such trips.
4. Create a scatterplot:
   - x-axis: `trip_distance`
   - y-axis: `fare_amount`

4. Briefly interpret the observed patterns

In [20]:
def haversine_distance(lat1, lon1, lat2, lon2):
    # Earth radius in miles
    R = 3958.8 
    
    # Haversine formula as a Polars expression
    dlat = (pl.col(lat2) - pl.col(lat1)).radians()
    dlon = (pl.col(lon2) - pl.col(lon1)).radians()
    
    a = (
        (dlat / 2).sin()**2 + 
        pl.col(lat1).radians().cos() * pl.col(lat2).radians().cos() * (dlon / 2).sin()**2
    )
    
    c = 2 * a.sqrt().arcsin()
    return R * c


In [21]:
cond_1 = pl.col("fare_amount") >= pl.col("fare_amount").quantile(0.9)
cond_2 = pl.col("tip_amount") > pl.col("fare_amount") * 0.5
cond_3 = (pl.col("trip_distance") < 2) & (pl.col("fare_amount") > pl.col("fare_amount").median())

ex7_df = df_filtered.with_columns(
    (cond_1).alias("c1"),
    (cond_2).alias("c2"),
    (cond_3).alias("c3")
)

high_value_trips_df = ex7_df.filter(
    (pl.col("c1").cast(pl.Int8) + 
     pl.col("c2").cast(pl.Int8) + 
     pl.col("c3").cast(pl.Int8)) >= 2
)

high_value_trips_df = high_value_trips_df.with_columns(
    haversine_distance(
        "pickup_latitude", 
        "pickup_longitude", 
        "dropoff_latitude", 
        "dropoff_longitude"
    ).alias("air_distance")
)

more_ten_miles_trips_df = high_value_trips_df.filter(
    pl.col("air_distance") > 10
)

more_ten_miles_trips_len = more_ten_miles_trips_df.height

print(f"#ROWS WITH AIR DISTANCE > 10 MILES: {more_ten_miles_trips_len}")

#ROWS WITH AIR DISTANCE > 10 MILES: 4666


### Spatial Analysis Results

| Metric | Value |
| :--- | :--- |
| **High-Value Trips > 10 Miles (Straight Line)** | 4,666 |


In [ ]:


scatterplot = more_ten_miles_trips_df.hvplot.scatter(
    x='trip_distance', 
    y='fare_amount',
    title='High-Value Trips: Fare Amount vs. Trip Distance',
    xlabel='Trip Distance (miles)',
    ylabel='Fare Amount ($)',
    alpha=0.5,       # Transparency helps visualize density
    size=10,         # Size of the dots
    color='teal',   
    width=700,
    height=500,
    grid=True,
    hover_cols=['air_distance'] # Shows the straight-line distance on hover
)

# Display the plot
scatterplot


:Scatter   [trip_distance]   (fare_amount,air_distance)

### Interpretation of Observed Patterns

Based on the scatterplot analysis, several distinct patterns emerge:

*   **Positive Correlation**: There is a clear linear relationship between `trip_distance` and `fare_amount`, confirming that fare prices generally increase in proportion to the distance traveled.
*   **Short-Distance Outliers**: A significant cluster of trips covers very short distances (between 0 and 1 mile) yet commands disproportionately high fares. This suggests these trips occur in high-demand or premium-priced areas, or perhaps involve significant surcharges.
*   **Fixed-Rate Plateaus**: A noticeable horizontal trend exists where the fare remains constant (at approximately **$50**) regardless of the trip distance. This likely indicates fixed-rate conventions, such as flat fares or specific pricing caps.


### Exercise 8 (16 pts)

Analyze driver performance using earnings efficiency:

1. For each trip, compute:
   - trip duration in hours.
   - total earnings = `fare_amount + tip_amount`.

2. Filter:
   - only keep durations between `3 minutes and 5 hours`.

3. For each driver (`hack_license`), compute:
   - total earnings.
   - total driving time (in hours).
   - earnings per hour.
   - total number of trips.

4. Select the **top 15% drivers** based on number of trips.

5. Classify trips into:
   - **day** (i.e., `06:00–18:00`).
   - **night** (remaining hours).

6. Compare driver efficiency:
   - Plot the distribution of earnings per hour for `day vs night drivers` (notice that a driver can be both a "day" and "night" driver in case it performed at least one day ride and one night ride).

7. Answer:
   - Which group appears more efficient?
   - Provide a short explanation based on your results.

In [ ]:
from numpy import sort

ex8_df = df_filtered.with_columns( 
    (pl.col("trip_time_in_secs") / 3600.0).alias("trip_time_in_hours"),
    (pl.col("trip_time_in_secs") / 60.0).alias("trip_time_in_minutes"),
    (pl.col("fare_amount") + pl.col("tip_amount")).alias("total_earnings")
).filter(
    (pl.col("trip_time_in_minutes") >= 3) & (pl.col("trip_time_in_hours") <= 5)
)
efficient_drivers_df = (
    ex8_df.group_by("hack_license")
    .agg([
        pl.sum("total_earnings").alias("total_earnings_sum"),
        pl.sum("trip_time_in_hours").alias("total_trip_time_hours"),
        pl.len().alias("total_trips")
    ])
    .with_columns(
        (pl.col("total_earnings_sum") / pl.col("total_trip_time_hours")).alias("earnings_per_hour")
    )
    .sort("earnings_per_hour", descending=True)
)
top_drivers_df = efficient_drivers_df.filter(
    pl.col("total_trips") >= pl.col("total_trips").quantile(0.85)
)

top_licenses = top_drivers_df.select("hack_license")

day_night_class_df = ex8_df.join(top_licenses, on="hack_license", how="semi").with_columns(
    pl.when(pl.col("pickup_datetime").dt.hour().is_between(6, 17))
    .then(pl.lit("day"))
    .otherwise(pl.lit("night"))
    .alias("time_of_day")
)
efficiency_comparison_df = (
    day_night_class_df.group_by(["hack_license", "time_of_day"])
    .agg([
        pl.sum("total_earnings").alias("total_earnings_sum"),
        pl.sum("trip_time_in_hours").alias("total_trip_time_hours")
    ])
    .with_columns(
        (pl.col("total_earnings_sum") / pl.col("total_trip_time_hours")).alias("earnings_per_hour")
    )
)

In [24]:
efficiency_comparison_df.filter(pl.col("earnings_per_hour") < 200).hvplot.violin(
    y='earnings_per_hour', 
    by='time_of_day', 
    title='Earnings Per Hour Distribution (Violin Plot): Day vs. Night',
    ylabel='Earnings per Hour ($)',
    height=450,
    width=600,
    cmap=['#f39c12', '#2c3e50']
).opts(active_tools=[])

:Violin   [time_of_day]   (earnings_per_hour)

### Efficiency Analysis: Day vs. Night Drivers

A **Violin Plot** was chosen for this comparison because it provides a more comprehensive view than a standard boxplot. While a boxplot only shows summary statistics (like the median and quartiles), the violin plot displays the **full probability density** of the data. This allows us to visualize not only the range of earnings but also where the majority of drivers are concentrated in terms of volume.

**Key Findings:**
*   **Superior Efficiency at Night**: The analysis indicates that night drivers are generally more efficient than day drivers. This is evidenced by the "bulge" of the violin being positioned higher on the Y-axis for the night category, meaning the majority of nighttime earnings per hour are higher than the majority of daytime earnings.
*   **Higher Peak Performance**: The nighttime distribution features longer and higher "tails," representing outliers with significantly higher maximum earnings compared to the daytime shift.
*   **Reduced Traffic Impact**: These patterns likely result from more favorable driving conditions at night (less congestion allowing for higher speeds) combined with potential nighttime surcharges and higher tipping behavior.
